In [2]:
import json
import pickle
import numpy as np
import cx_Oracle
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

import qlib
from qlib.data import D
import pandas as pd
import numpy as np


In [31]:
data_path="/home/quant/zc/finance_deal/qlib_data/price_data0821"
qlib.init(provider_uri=data_path)

[2631286:MainThread](2026-03-31 13:34:02,907) INFO - qlib.Initialization - [config.py:420] - default_conf: client.
[2631286:MainThread](2026-03-31 13:34:03,396) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[2631286:MainThread](2026-03-31 13:34:03,399) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/home/quant/zc/finance_deal/qlib_data/price_data0821')}


In [440]:
baijiu = D.features(
    instruments=['399997.SZ'],
    fields=['$close'],
    # start_time="2024-01-01",
    # end_time="2020-01-01",
    freq='day',
)

In [442]:
baijiu=baijiu.reset_index()
baijiu.rename(columns={'$close':'close'},inplace=True)
baijiu

,instrument,datetime,close
0,399997.SZ,2010-01-04,2488.445068
1,399997.SZ,2010-01-05,2486.956055
2,399997.SZ,2010-01-06,2442.008057
3,399997.SZ,2010-01-07,2377.396973
4,399997.SZ,2010-01-08,2401.623047
...,...,...,...
3931,399997.SZ,2026-03-17,8283.671875
3932,399997.SZ,2026-03-18,8178.873047
3933,399997.SZ,2026-03-19,8048.816406
3934,399997.SZ,2026-03-20,7976.902832


In [412]:
calendar = D.calendar(
                start_time="2016-07-01",
                end_time="2026-01-01",
                freq='day'
            )

In [413]:
calendar

array([Timestamp('2016-07-01 00:00:00'), Timestamp('2016-07-04 00:00:00'),
       Timestamp('2016-07-05 00:00:00'), ...,
       Timestamp('2025-12-29 00:00:00'), Timestamp('2025-12-30 00:00:00'),
       Timestamp('2025-12-31 00:00:00')], dtype=object)

In [414]:
data_df=pd.read_csv("back_tests_new.csv")

In [415]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 假设 df 是你的 DataFrame
df = data_df

# 转换日期
df['日期'] = pd.to_datetime(df['日期'])

# 选择数值列
feature_cols = ['未知区间', '上升区间', '下降区间', '峰点', '谷点']

# Min-Max 归一化（处理全零列）
scaler = MinMaxScaler()
df_norm = df.copy()

for col in feature_cols:
    if df[col].max() == df[col].min():  # 全零或常数
        df_norm[col] = 0.0
    else:
        df_norm[col] = scaler.fit_transform(df[[col]])

print(df_norm)

            日期  未知区间      上升区间      下降区间        峰点        谷点
0   2016-07-31   0.0  0.000000  0.000000  0.000000  0.000000
1   2016-08-31   0.0  0.000000  0.000000  0.000000  0.000000
2   2016-09-30   0.0  0.000000  0.000000  0.000000  0.000000
3   2016-10-31   0.0  0.000000  0.066667  0.000000  0.000000
4   2016-11-30   0.0  0.000000  0.066667  0.000000  0.000000
..         ...   ...       ...       ...       ...       ...
108 2025-07-31   0.0  0.260870  0.133333  0.208333  0.000000
109 2025-08-31   0.0  0.043478  0.200000  0.208333  0.176471
110 2025-09-30   0.0  0.130435  0.400000  0.250000  0.000000
111 2025-10-31   0.0  0.086957  0.400000  0.000000  0.058824
112 2025-11-30   0.0  0.260870  0.133333  0.000000  0.000000

[113 rows x 6 columns]


In [416]:
date_select=-12
dis_datetime=data_df[data_df.columns[0]].values
features=data_df[data_df.columns[1:]].values
buy_w=np.array([0,0.4,0,0,0.6])
sell_w=np.array([0,0,0.4,0.6,0])
buy_signal=[]
sell_signal=[]
for f in features:
    sell_signal.append((f*sell_w).sum())
    buy_signal.append((f*buy_w).sum())
sell_signal=np.array(sell_signal)
buy_signal=np.array(buy_signal)

In [417]:
data_df['buy_signal']=buy_signal
data_df['sell_signal']=sell_signal

In [418]:
data_df.rename(columns={'日期':'datetime'},inplace=True)
data_df

,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal
0,2016-07-31,0,0,0,0,0,0.0,0.0
1,2016-08-31,0,0,0,0,0,0.0,0.0
2,2016-09-30,0,0,0,0,0,0.0,0.0
3,2016-10-31,0,0,1,0,0,0.0,0.4
4,2016-11-30,0,0,1,0,0,0.0,0.4
...,...,...,...,...,...,...,...,...
108,2025-07-31,0,6,2,5,0,2.4,3.8
109,2025-08-31,0,1,3,5,3,2.2,4.2
110,2025-09-30,0,3,6,6,0,1.2,6.0
111,2025-10-31,0,2,6,0,1,1.4,2.4


In [419]:
start_time=min(data_df['datetime'])
end_time=max(data_df['datetime'])

In [420]:
date_df = pd.read_csv(
        "/home/quant/zc/finance_deal/qlib_data/price_data0821/calendars/day.txt",
        sep='\s+',
        header=None,
        names=['col1']
    )

In [421]:
date_df.rename(columns={'col1':'datetime'},inplace=True)

In [422]:
date_df

,datetime
0,2005-01-04
1,2005-01-05
2,2005-01-06
3,2005-01-07
4,2005-01-10
...,...
5144,2026-03-13
5145,2026-03-16
5146,2026-03-17
5147,2026-03-18


In [423]:
data_df['datetime'].isin(date_df['datetime'])
data_df['datetime']=pd.to_datetime(data_df['datetime'])
date_df['datetime']=pd.to_datetime(date_df['datetime'])

In [424]:
aligned_df = pd.merge_asof(date_df, data_df, on='datetime', direction='backward')

In [425]:
aligned_df=aligned_df.dropna()  

In [426]:
# 方法1b：如果交易日历可能月末那天不是交易日，使用月份最后一天
month_end_df = (
    aligned_df
    .groupby(aligned_df['datetime'].dt.to_period('M'))
    .last()  # 取每个月最后一个交易日
)

In [427]:
month_end_df=month_end_df.reset_index(drop=True)

In [428]:
month_end_df

,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal
0,2016-08-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2016-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2016-10-31,0.0,0.0,1.0,0.0,0.0,0.0,0.4
3,2016-11-30,0.0,0.0,1.0,0.0,0.0,0.0,0.4
4,2016-12-30,0.0,0.0,1.0,0.0,0.0,0.0,0.4
...,...,...,...,...,...,...,...,...
111,2025-11-28,0.0,2.0,6.0,0.0,1.0,1.4,2.4
112,2025-12-31,0.0,6.0,2.0,0.0,0.0,2.4,0.8
113,2026-01-30,0.0,6.0,2.0,0.0,0.0,2.4,0.8
114,2026-02-27,0.0,6.0,2.0,0.0,0.0,2.4,0.8


In [429]:
def map_monthly_signals_to_trading_days(date_df, month_end_df):
    """
    将月频信号映射到交易日上，只在月末交易日有信号值
    
    Parameters:
    -----------
    date_df : pandas.DataFrame
        交易日DataFrame，包含一列日期，5147行
    month_end_df : pandas.DataFrame
        月频信号DataFrame，76行×3列（日期、buy_signal、sell_signal）
    
    Returns:
    --------
    result_df : pandas.DataFrame
        5147行×3列（日期、buy_signal、sell_signal），只在月末交易日有信号值
    """
    
    # 确保日期列都是datetime类型
    date_df = date_df.copy()
    date_df.columns = ['datetime']  # 假设第一列是日期，确保列名为'date'
    date_df['datetime'] = pd.to_datetime(date_df['datetime'])
    
    month_end_df = month_end_df.copy()
    month_end_df['datetime'] = pd.to_datetime(month_end_df['datetime'])
    
    # 方法1：使用merge直接匹配（推荐）
    # 将月频信号左连接到交易日DataFrame
    result_df = pd.merge(
        date_df,
        month_end_df,
        on='datetime',
        how='left'
    )

    
    return result_df

date_df1=date_df[date_df['datetime']>=start_time]
mapped_signals = map_monthly_signals_to_trading_days(date_df1, month_end_df)
mapped_signals


,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal
0,2016-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-08-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-08-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2332,2026-03-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2333,2026-03-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2334,2026-03-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2335,2026-03-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [430]:
def add_trading_columns(mapped_signals_df):
    """
    为映射后的信号添加action和weight列
    
    Parameters:
    -----------
    mapped_signals_df : pandas.DataFrame
        映射后的信号DataFrame，包含date、buy_signal、sell_signal三列
    
    Returns:
    --------
    result_df : pandas.DataFrame
        添加了action和weight列的DataFrame
    """
    
    # 复制数据
    df = mapped_signals_df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # 初始化新列
    df['action'] = 0      # 0:无操作, 1:买入, -1:卖出
    df['weight'] = 0.0    # 权重
    
    n = len(df)
    i = 0
    
    while i < n:
        # 检查买入条件
        if (pd.notna(df.loc[i, 'buy_signal']) and 
            pd.notna(df.loc[i, 'sell_signal']) and 
            df.loc[i, 'buy_signal'] > 0 and 
            df.loc[i, 'buy_signal'] > df.loc[i, 'sell_signal']):
        # if (pd.notna(df.loc[i, 'buy_signal']) and 
        #     pd.notna(df.loc[i, 'sell_signal']) and 
        #     df.loc[i, 'buy_signal'] >0 ):
            # 买入信号
            df.loc[i, 'action'] = 1
            df.loc[i, 'weight'] = 1.0  # 满仓
            
            # 20个交易日后的卖出信号
            sell_idx = i + 20
            if sell_idx < n:
                df.loc[sell_idx, 'action'] = -1
                df.loc[sell_idx, 'weight'] = 1.0
            
            # 跳过持股期间
            i = sell_idx + 1
        else:
            i += 1
    
    return df


In [431]:
def add_trading_columns1(mapped_signals_df):
    """
    为映射后的信号添加action和weight列，并去除连续买入信号（只保留首次买入）
    
    Parameters:
    -----------
    mapped_signals_df : pandas.DataFrame
        映射后的信号DataFrame，包含datetime、buy_signal、sell_signal三列
    
    Returns:
    --------
    result_df : pandas.DataFrame
        添加了action和weight列的DataFrame
    """
    df = mapped_signals_df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)  # 确保按时间排序

    # 初始化新列
    df['action'] = 0      # 0:无操作, 1:买入, -1:卖出
    df['weight'] = 0.0    # 权重

    in_position = False  # 状态标志：是否已持仓

    for i in range(len(df)):
        buy_val = df.loc[i, 'buy_signal']
        sell_val = df.loc[i, 'sell_signal']

        # 确保信号非空
        has_buy = pd.notna(buy_val) and buy_val > 0
        has_sell = pd.notna(sell_val) and sell_val > 0

        if in_position:
            # 已持仓：只响应卖出信号
            if has_sell and (not has_buy or sell_val > buy_val):
                df.loc[i, 'action'] = -1
                df.loc[i, 'weight'] = 1.0
                in_position = False  # 卖出后清仓
            # 否则：忽略买入信号（即使 buy > sell）
        else:
            # 未持仓：只响应买入信号
            if has_buy and (not has_sell or buy_val > sell_val):
                df.loc[i, 'action'] = 1
                df.loc[i, 'weight'] = 1.0
                in_position = True   # 买入后建仓
            # 否则：忽略卖出信号（空仓不能卖）

    return df

In [432]:
def add_trading_columns2(mapped_signals_df):
    """
    为映射后的信号添加action和weight列，并去除连续买入信号（只保留首次买入）
    
    Parameters:
    -----------
    mapped_signals_df : pandas.DataFrame
        映射后的信号DataFrame，包含datetime、buy_signal、sell_signal三列
    
    Returns:
    --------
    result_df : pandas.DataFrame
        添加了action和weight列的DataFrame
    """
    df = mapped_signals_df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)  # 确保按时间排序

    # 初始化新列
    df['action'] = 0      # 0:无操作, 1:买入, -1:卖出
    df['weight'] = 0.0    # 权重

    in_position = False  # 状态标志：是否已持仓

    for i in range(len(df)):
        buy_val = df.loc[i, 'buy_signal']
        sell_val = df.loc[i, 'sell_signal']

        # 确保信号非空
        has_buy = pd.notna(buy_val) and buy_val > 0
        has_sell = pd.notna(sell_val) and sell_val > 0

        if in_position:
            # 已持仓：只响应卖出信号
            if has_sell:
                df.loc[i, 'action'] = -1
                df.loc[i, 'weight'] = 1.0
                in_position = False  # 卖出后清仓
            # 否则：忽略买入信号（即使 buy > sell）
        else:
            # 未持仓：只响应买入信号
            if has_buy and (not has_sell or buy_val > sell_val):
                df.loc[i, 'action'] = 1
                df.loc[i, 'weight'] = 1.0
                in_position = True   # 买入后建仓
            # 否则：忽略卖出信号（空仓不能卖）

    return df

In [433]:

result = add_trading_columns1(mapped_signals)
result

,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal,action,weight
0,2016-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
1,2016-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
2,2016-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
3,2016-08-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
4,2016-08-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
...,...,...,...,...,...,...,...,...,...,...
2332,2026-03-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
2333,2026-03-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
2334,2026-03-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
2335,2026-03-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0


In [434]:
# result['action']=0
# result['weight']=0
# result.loc[0,'action']=1
# result.loc[0,'weight']=1
# result

In [435]:
result['instrument']='603369.SH'
first_nonzero=(result['action']!=0).idxmax()
result1=result[first_nonzero-2:]
result1

,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal,action,weight,instrument
159,2017-03-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
160,2017-03-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
161,2017-03-31,0.0,13.0,5.0,0.0,0.0,5.2,2.0,1,1.0,603369.SH
162,2017-04-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
163,2017-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
...,...,...,...,...,...,...,...,...,...,...,...
2332,2026-03-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
2333,2026-03-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
2334,2026-03-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH
2335,2026-03-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,603369.SH


In [436]:
result1.to_csv("trade_260317.csv",index=False)

In [437]:
result2=result1.dropna()
result2

,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal,action,weight,instrument
161,2017-03-31,0.0,13.0,5.0,0.0,0.0,5.2,2.0,1,1.0,603369.SH
179,2017-04-28,0.0,13.0,5.0,0.0,0.0,5.2,2.0,0,0.0,603369.SH
199,2017-05-31,0.0,22.0,4.0,0.0,0.0,8.8,1.6,0,0.0,603369.SH
221,2017-06-30,0.0,20.0,0.0,0.0,0.0,8.0,0.0,0,0.0,603369.SH
242,2017-07-31,0.0,18.0,0.0,0.0,0.0,7.2,0.0,0,0.0,603369.SH
...,...,...,...,...,...,...,...,...,...,...,...
2265,2025-11-28,0.0,2.0,6.0,0.0,1.0,1.4,2.4,0,0.0,603369.SH
2288,2025-12-31,0.0,6.0,2.0,0.0,0.0,2.4,0.8,1,1.0,603369.SH
2308,2026-01-30,0.0,6.0,2.0,0.0,0.0,2.4,0.8,0,0.0,603369.SH
2322,2026-02-27,0.0,6.0,2.0,0.0,0.0,2.4,0.8,0,0.0,603369.SH


In [405]:
result3=result1.dropna()
result3

,datetime,未知区间,上升区间,下降区间,峰点,谷点,buy_signal,sell_signal,action,weight,instrument
120,2017-01-26,0.0,1.0,0.0,0.0,0.0,0.4,0.0,1,1.0,603369.SH
138,2017-02-28,0.0,1.0,4.0,0.0,0.0,0.4,1.6,-1,1.0,603369.SH
161,2017-03-31,0.0,6.0,8.0,0.0,0.0,2.4,3.2,0,0.0,603369.SH
179,2017-04-28,0.0,6.0,8.0,0.0,0.0,2.4,3.2,0,0.0,603369.SH
199,2017-05-31,0.0,14.0,1.0,0.0,0.0,5.6,0.4,1,1.0,603369.SH
...,...,...,...,...,...,...,...,...,...,...,...
2265,2025-11-28,0.0,0.0,41.0,0.0,0.0,0.0,16.4,0,0.0,603369.SH
2288,2025-12-31,0.0,4.0,10.0,0.0,0.0,1.6,4.0,0,0.0,603369.SH
2308,2026-01-30,0.0,4.0,10.0,0.0,0.0,1.6,4.0,0,0.0,603369.SH
2322,2026-02-27,0.0,4.0,10.0,0.0,0.0,1.6,4.0,0,0.0,603369.SH


In [250]:
s1=result[result['datetime']=='2025-07-31']

In [251]:
s1['buy_signal'].iloc[0]

18.2

In [4]:
# stock_map = {
#     "五粮液": "000858.SZ",
#     "贵州茅台": "600519.SH",
#     "今世缘": "603369.SH",
#     "泸州老窖": "000568.SZ",
#     "古井贡酒": "000596.SZ",
#     "口子窖": "603589.SH",
#     "老白干酒": "600559.SH",
#     "山西汾酒": "600809.SH",
#     "水井坊": "600779.SH",
#     "迎驾贡酒": "603198.SH",
#     "洋河股份": "002304.SZ",
#     "伊力特": "600197.SH",
# }
stock_map={
    "比亚迪":"002594.SZ",
    "宁德时代": "300750.SZ",
    "欣旺达": "300207.SZ",
    "亿纬锂能": "300014.SZ",
    "孚能科技": "688567.SH",
    "珠海冠宇": "688772.SH",
}

from pathlib import Path
input_root=Path("/home/quant/zc/backtrader/QuantBacktester_57/llm/input")
for json_file in sorted(input_root.glob("*/dis_output_test_test_v4.json")):
    folder_name = json_file.parent.name
    if folder_name in stock_map:
        logic_list=[]
        with open(json_file,'r',encoding='utf-8') as file:
            data=json.load(file)
            data=data.get("kgData")['nodes']
            for node in data:
                if node["type"]=="逻辑":
                    logic_list.append(node)
        with open(f'{json_file.parent}/strategy_clues.json','w',encoding='utf-8') as f:
            json.dump(logic_list,f,ensure_ascii=False,indent=4)

In [26]:
json_file.parent

PosixPath('/home/quant/zc/backtrader/QuantBacktester_57/llm/input/迎驾贡酒')

In [23]:
with open('strategy_clues.json','w',encoding='utf-8') as f:
    json.dump(logic_list,f,ensure_ascii=False,indent=4)

In [124]:
import pandas as pd
import numpy as np

def evaluate_factor(signals: pd.DataFrame, prices: pd.DataFrame, n_quantiles: int = 5):
    # 1) 标准化字段
    s = signals.copy()
    p = prices.copy()
    s["datetime"] = pd.to_datetime(s["datetime"])
    p["datetime"] = pd.to_datetime(p["datetime"])

    # 2) 构造 score（你可替换成任一原始列，如 s["谷点"]）
    # s["score"] = s["buy_signal"] - s["sell_signal"]

    # 3) 月末价格 -> 未来1M收益
    p = p.sort_values(["instrument", "datetime"])
    p["month"] = p["datetime"].dt.to_period("M")
    month_close = (
        p.groupby(["instrument", "month"], as_index=False)
         .last()[["instrument", "month", "datetime", "close"]]
         .rename(columns={"datetime": "month_end_dt", "close": "month_close"})
    )
    month_close["future_ret_1m"] = (
        month_close.groupby("instrument")["month_close"].shift(-1) / month_close["month_close"] - 1.0
    )

    # 4) 信号按月对齐（默认信号已是月频；若日频则取月末最后一条）
    s["month"] = s["datetime"].dt.to_period("M")
    month_signal = (
        s.sort_values(["instrument", "datetime"])
         .groupby(["instrument", "month"], as_index=False)
         .last()[["instrument", "month", "score"]]
    )

    df = month_signal.merge(
        month_close[["instrument", "month", "future_ret_1m"]],
        on=["instrument", "month"],
        how="inner"
    ).dropna(subset=["score", "future_ret_1m"])

    # 5) IC（按月横截面）
    def cs_ic(g):
        if g["score"].nunique() < 2 or g["future_ret_1m"].nunique() < 2:
            return np.nan
        return g["score"].corr(g["future_ret_1m"], method="spearman")

    ic_by_month = df.groupby("month").apply(cs_ic).rename("ic").dropna()

    ic_mean = ic_by_month.mean() if len(ic_by_month) else np.nan
    ic_ir = ic_mean / ic_by_month.std(ddof=1) if len(ic_by_month) > 1 and ic_by_month.std(ddof=1) != 0 else np.nan

    # 6) 分层（按月分位）
    def add_bucket(g):
        if g["score"].nunique() < n_quantiles:
            g["bucket"] = np.nan
            return g
        g["bucket"] = pd.qcut(g["score"], q=n_quantiles, labels=False, duplicates="drop") + 1
        return g

    bdf = df.groupby("month", group_keys=False).apply(add_bucket).dropna(subset=["bucket"])
    bdf["bucket"] = bdf["bucket"].astype(int)

    layer_ret = (
        bdf.groupby(["month", "bucket"], as_index=False)["future_ret_1m"]
           .mean()
           .pivot(index="month", columns="bucket", values="future_ret_1m")
           .sort_index(axis=1)
    )

    long_short = None
    if len(layer_ret.columns) >= 2:
        long_short = layer_ret[layer_ret.columns.max()] - layer_ret[layer_ret.columns.min()]

    result = {
        "sample_count": len(df),
        "months_count": df["month"].nunique(),
        "ic_mean": ic_mean,
        "ic_ir": ic_ir,
        "ic_by_month": ic_by_month,
        "layer_ret": layer_ret,
        "long_short_mean": long_short.mean() if long_short is not None else np.nan,
        "long_short_win_rate": (long_short > 0).mean() if long_short is not None else np.nan,
    }
    return result

In [125]:
prices = D.features(
                instruments=["600519.SH","000858.SZ","603369.SH","000568.SZ","000596.SZ","603589.SH","600559.SH","600809.SH","600779.SH","603198.SH","002304.SZ","600197.SH"],
                fields=['$close'],
                start_time='2024-10-30',
                end_time='2026-01-01',
                freq='day',
            ).reset_index()
prices.rename(columns={'$close': 'close'}, inplace=True)

In [126]:
signals=pd.read_csv("/home/quant/zc/backtrader/QuantBacktester_57/llm/trade_portfolio_20_0.csv")

In [127]:
evaluate_factor(signals,prices)

{'sample_count': 54,
 'months_count': 14,
 'ic_mean': -0.0346938775510204,
 'ic_ir': -0.05628621643146149,
 'ic_by_month': month
 2024-10    0.100000
 2024-11    0.500000
 2024-12    0.500000
 2025-01   -0.800000
 2025-02   -1.000000
 2025-03    0.800000
 2025-04    0.400000
 2025-05    0.200000
 2025-06    0.200000
 2025-07   -0.800000
 2025-08   -0.400000
 2025-09   -0.485714
 2025-10    0.800000
 2025-11   -0.500000
 Freq: M, Name: ic, dtype: float64,
 'layer_ret': bucket          1         2         3         4         5
 month                                                    
 2024-10  0.006475  0.060570  0.046169  0.023679  0.024260
 2025-09  0.024875 -0.018092 -0.020497 -0.012214 -0.017442,
 'long_short_mean': -0.012266278,
 'long_short_win_rate': 0.5}

In [129]:
prices[prices['instrument']=="000568.SZ"]

,instrument,datetime,close
0,000568.SZ,2024-10-30,4905.128906
1,000568.SZ,2024-10-31,4858.519531
2,000568.SZ,2024-11-01,4895.090332
3,000568.SZ,2024-11-04,4922.697266
4,000568.SZ,2024-11-05,5059.658203
...,...,...,...
283,000568.SZ,2025-12-25,4541.706543
284,000568.SZ,2025-12-26,4491.669434
285,000568.SZ,2025-12-29,4409.653809
286,000568.SZ,2025-12-30,4433.731934


In [135]:
def daily_to_monthly_close(df: pd.DataFrame, date_col: str = 'datetime', close_col: str = 'close') -> pd.Series:
    """
    将日线DataFrame（含datetime列和close列）转换为月线收盘价序列（取月末最后一个交易日价格）。

    参数:
    df (pd.DataFrame): 输入的日线数据，需包含日期列和收盘价列。
    date_col (str): 日期列的列名，默认为'datetime'。
    close_col (str): 收盘价列的列名，默认为'close'。

    返回:
    pd.Series: 月线收盘价序列，索引为月末日期，值为对应月末最后一个交易日的收盘价。
    """
    # 检查输入是否为DataFrame
    if not isinstance(df, pd.DataFrame):
        raise TypeError("输入必须是pd.DataFrame类型")
    # 检查必要列是否存在
    missing_cols = [col for col in [date_col, close_col] if col not in df.columns]
    if missing_cols:
        raise ValueError(f"DataFrame缺少必要列: {missing_cols}")
    # 复制数据避免修改原DataFrame
    df_copy = df[[date_col, close_col]].copy()
    # 将日期列转换为datetime格式
    df_copy[date_col] = pd.to_datetime(df_copy[date_col], errors='coerce')
    # 删除日期转换失败的行（NaT）
    df_valid = df_copy.dropna(subset=[date_col])
    if df_valid.empty:
        raise ValueError("转换后无有效日期数据")
    # 设置日期为索引
    df_valid.set_index(date_col, inplace=True)
    # 按月重采样，取最后一个交易日的收盘价
    monthly_close = df_valid[close_col].resample('M').last().dropna()
    return monthly_close

In [136]:
t1=daily_to_monthly_close(prices[prices['instrument']=="000568.SZ"])

In [137]:
t1

datetime
2024-10-31    4858.519531
2024-11-30    4973.967773
2024-12-31    4488.869141
2025-01-31    4191.799316
2025-02-28    4587.580566
2025-03-31    4705.843262
2025-04-30    4488.907227
2025-05-31    4241.499023
2025-06-30    4113.804199
2025-07-31    4458.434570
2025-08-31    5161.337891
2025-09-30    4963.070801
2025-10-31    5061.640137
2025-11-30    5112.053223
2025-12-31    4372.408203
Freq: M, Name: close, dtype: float32

In [5]:
11407.828125/12376.905273-1

-0.07829720973255128

In [9]:
def sigmoid_basic(x):
    return 1/(1+np.exp(-x))
result=sigmoid_basic(pd.Series([-0.0801,-0.4182]))

In [10]:
result

0    0.479986
1    0.396948
dtype: float64